# เฟส 1 — ตรวจ dBZ stack ที่จะป้อน pystepsnotebook นี้เป็นแค่**ตัวเรียกใช้** ตรรกะจริงอยู่ใน `radar_archive/build_stack.py` และ `grid.py` ทั้งหมดใช้ดูว่า stack ที่สร้างออกมาถูกต้องก่อนเอาไปทำ nowcastรันบน Google Colab ได้เลย

In [ ]:
# --- ติดตั้ง ---!git clone -q https://github.com/jamorn12/tmd-radar-archive.git%cd tmd-radar-archive!pip install -q -r requirements.txt!apt-get install -y -qq tesseract-ocr > /dev/nullprint("พร้อม")

## 1. ดูว่ามีช่วงเวลาต่อเนื่องอะไรบ้าง (ยังไม่สร้างไฟล์)

In [ ]:
!python -m radar_archive.build_stack --station PHS --list

## 2. สร้าง stack จริง`--agg mean` เฉลี่ยใน **linear Z** ไม่ใช่ใน dBZ — dBZ เป็นสเกลลอการิทึมถ้าเฉลี่ยตรง ๆ จะได้ค่าต่ำกว่าความจริงเสมอ`--agg max` เก็บแกนฝนไว้ครบกว่า เหมาะถ้าสนใจค่าสุดขีด

In [ ]:
!python -m radar_archive.build_stack --station PHS --agg mean

## 3. โหลด stack ที่ยาวที่สุดมาดู

In [ ]:
import json, numpy as np, matplotlib.pyplot as pltfrom pathlib import Pathfrom radar_archive import gridman = json.loads(Path('data/stack/manifest.json').read_text())man.sort(key=lambda m: -m['shape'][0])for m in man:    print(f"{m['name']:<28}{m['shape'][0]:>3} เฟรม  cover {m['check']['cover_mean']}%  "          f"dBZ {m['check']['dbz_min']}-{m['check']['dbz_max']}  "          f"{'มีข้อสังเกต' if m['issues'] else 'ผ่าน'}")pick = man[0]stack, times = grid.load_stack(Path('data/stack') / pick['file'])print(f"\nเลือก {pick['name']}  shape {stack.shape}")

## 4. ตรวจสามอย่างที่ถ้าผิดแล้ว pysteps จะไม่ฟ้อง1. **เฟรมห่างเท่ากันจริงไหม** — ถ้าไม่ ความเร็วที่คำนวณได้จะผิดตามสัดส่วน2. **NaN กับ 0 ปนกันหรือเปล่า** — NaN ต้องแปลว่า *ไม่มีข้อมูล* เท่านั้น ส่วน *ไม่มีฝน* ต้องเป็น 03. **แถวที่ 0 อยู่ใต้สุดจริงไหม** — ถ้าสลับ แกน v ของสนามลมจะกลับทิศเงียบ ๆ

In [ ]:
ok, gaps = grid.check_regular(times)print("1. ช่วงห่างระหว่างเฟรม (นาที):", [round(g,1) for g in gaps], "->", "ผ่าน" if ok else "ไม่ผ่าน")e, n = grid.grid_km()EE, NN = np.meshgrid(e, n)inside = np.hypot(EE, NN) <= grid.GRID_HALF_KMprint(f"2. ในวง 240 กม.: NaN {100*np.mean(~np.isfinite(stack[:,inside])):.3f}%  "      f"(ควรใกล้ 0) · นอกวง: finite {np.isfinite(stack[:,~inside]).sum()} เซลล์ (ควรเป็น 0)")# แถวล่างสุดต้องอยู่ใต้สุด -> ค่า N ของแถว 0 ต้องติดลบprint(f"3. แถวที่ 0 อยู่ที่ N = {n[0]:+.0f} กม. -> {'ใต้สุด ถูกต้อง' if n[0] < 0 else 'ผิด! กลับด้านอยู่'}")

## 5. ดูภาพทุกเฟรมในช่วงนี้

In [ ]:
from matplotlib.colors import BoundaryNorm, ListedColormapfrom radar_archive import palette as pallv = np.concatenate([[0.0], pal.TMD_REF_DBZ])cmap = ListedColormap(np.vstack([[0,0,0], pal.TMD_REF_RGB])/255.0)norm = BoundaryNorm(np.concatenate([lv, [70.0]]), cmap.N)T = stack.shape[0]cols = 5; rows = int(np.ceil(T/cols))fig, axes = plt.subplots(rows, cols, figsize=(3.0*cols, 3.1*rows), facecolor='white')for i, ax in enumerate(np.ravel(axes)):    ax.set_axis_off()    if i >= T: continue    ax.imshow(stack[i], origin='lower', cmap=cmap, norm=norm,              extent=[-240, 240, -240, 240], interpolation='nearest')    ax.add_patch(plt.Circle((0,0), 240, fill=False, ec='0.6', lw=.6))    ax.plot(0, 0, '+', color='0.3', ms=6)    ax.set_title(f"{times[i]:%H:%M}Z   max {np.nanmax(stack[i]):.1f} dBZ", fontsize=9)fig.suptitle(f"{pick['name']}  ({T} frames, 241x241 @ 2 km, aeqd)", y=1.0)fig.tight_layout(); plt.show()

## 6. ตรวจ motion แบบหยาบ ๆ ก่อนไปใช้ pystepsหา displacement ระหว่างคู่เฟรมท้าย ๆ ด้วย block matching **เฉพาะบริเวณที่มี echo**> phase correlation ใช้ไม่ได้กับ field ของเรา — echo มีไม่ถึง 1% ของภาพ> พื้นหลังศูนย์กลบสัญญาณจนได้ค่ามั่ว ทดสอบแล้วได้ (0,0) สลับกับ (-81,+49)> `dense_lucaskanade` ของ pysteps ก็ควรใส่ mask ด้วยเหตุผลเดียวกัน

In [ ]:
from scipy import ndimagedef field(f):    a = np.nan_to_num(f, nan=0.0)    return ndimage.gaussian_filter(np.clip(a, 0, None), 1.5)def best_shift(A, B, R=12):    m = (A > 1) | (B > 1)    if m.sum() < 30: return None    ys, xs = np.nonzero(m)    y0, x0 = max(ys.min()-R, R), max(xs.min()-R, R)    y1, x1 = min(ys.max()+R, A.shape[0]-R), min(xs.max()+R, A.shape[1]-R)    if y1 <= y0 or x1 <= x0: return None    Bc = B[y0:y1, x0:x1]; best = (np.inf, 0, 0)    for dy in range(-R, R+1):        for dx in range(-R, R+1):            e = float(((A[y0-dy:y1-dy, x0-dx:x1-dx] - Bc)**2).sum())            if e < best[0]: best = (e, dy, dx)    return best[1], best[2]KM = grid.KM_PER_PIXELprint("displacement ต่อ 15 นาที (แถว 0 = ใต้สุด ดังนั้น dy บวก = เคลื่อนขึ้นเหนือ)")vs = []for i in range(max(0, len(times)-4), len(times)-1):    s = best_shift(field(stack[i]), field(stack[i+1]))    if s is None: continue    dy, dx = s    spd = np.hypot(dy, dx)*KM*4    brg = (np.degrees(np.arctan2(dx, dy)) + 360) % 360    vs.append((dy, dx))    print(f"  {times[i]:%H:%M} -> {times[i+1]:%H:%M}  (dy,dx)=({dy:+3d},{dx:+3d})  "          f"{spd:5.1f} กม./ชม.  ไปทาง {brg:5.1f}°")if vs:    a = np.array(vs)    print(f"\nค่ากลาง: ({np.median(a[:,0]):+.0f},{np.median(a[:,1]):+.0f}) px/15min"          f"  = {np.hypot(*np.median(a,0))*KM*4:.1f} กม./ชม.")    print(f"ความนิ่ง: ขนาดต่างกัน {np.std(np.hypot(a[:,0],a[:,1])*KM*4):.1f} กม./ชม. "          f"(ยิ่งน้อยยิ่งเชื่อได้)")

## 7. ผลของการแก้ตาราง dBZค่า dBZ ของแต่ละแถบสีเดิมเราคำนวณเองแบบเชิงเส้นตอนนี้ใช้ตารางจริงจาก ONWR (สทนช.) ซึ่งใช้ข้อมูล TMD ชุดเดียวกัน

In [ ]:
from PIL import Imagefrom radar_archive.config import get_stationst = get_station("PHS")img = Image.open(sorted(Path('data/raw/PHS').rglob('*.jpg'))[-1]).convert('RGB')rgb, new = pal.extract_palette(img, st)_,   old = pal.extract_palette(img, st, use_reference=False)fig, ax = plt.subplots(figsize=(7.5, 4), facecolor='white')y = np.arange(len(new))ax.barh(y, new - old, color=['#c94f4f' if abs(v) > 1 else '#7f9ab5' for v in (new-old)])ax.set_yticks(y); ax.set_yticklabels([f"{o:.1f} -> {n:.1f}" for o, n in zip(old, new)], fontsize=7)ax.axvline(0, color='0.3', lw=.8)ax.set_xlabel("change in dBZ (new - old)")ax.set_title("Effect of using the authoritative TMD level table")ax.invert_yaxis(); fig.tight_layout(); plt.show()print("แถบที่เปลี่ยนเกิน 1 dBZ:")for c, o, n in zip(rgb, old, new):    if abs(n-o) > 1:        print(f"  RGB {str([int(v) for v in c]):<18} {o:5.1f} -> {n:5.1f}  ({n-o:+.1f})")

## สรุปถ้าทุกข้อข้างบนผ่าน = **stack พร้อมป้อน pysteps แล้ว**ขั้นต่อไป (เฟส 2) คือ `nowcast.py`:`dense_lucaskanade` -> ตรวจความนิ่งของ motion -> `extrapolate` 4 ขั้น -> เขียน PNG + `latest.json`> ข้อควรระวังที่เจอแล้ว: ตัวหนังสือสีขาวบนแผนที่ (ชื่อเมือง) ถ้าไปติดกับก้อนฝนจริง> จะถูกนับเป็น echo แรงสุด (เคยเจอ 59.2 dBZ ที่จริงคือคำว่า "Petchabun")> `build_stack` เตือนให้แล้วเมื่อเจอเซลล์แรงจัดโดด ๆ ไม่กี่เซลล์ในเฟรมเดียว> ต้องแก้ก่อนเชื่อค่า motion เต็มที่